In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
def lab_white_balance(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    avg_a = np.mean(a)
    avg_b = np.mean(b)

    a = a - ((avg_a - 128) * (l / 255.0))
    b = b - ((avg_b - 128) * (l / 255.0))

    lab = cv2.merge([l, a.astype(np.uint8), b.astype(np.uint8)])
    balanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    return balanced

In [3]:
def extract_skin_roi(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    lower_skin = np.array([0, 20, 70])
    upper_skin = np.array([20, 255, 255])

    mask = cv2.inRange(hsv, lower_skin, upper_skin)
    skin = cv2.bitwise_and(image, image, mask=mask)

    return skin

In [4]:
def build_pixel_sequence(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    rgb_flat = image.reshape(-1,3)
    hsv_flat = hsv.reshape(-1, 3)

    combined = np.concatenate([rgb_flat, hsv_flat], axis=1)

    return combined.astype(np.float32)

In [5]:
IMG_SIZE = 64

def preprocess_image(path):
    image = cv2.imread(path)

    if image is None:
        raise ValueError(f"Image not found: {path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    image = lab_white_balance(image)
    image = extract_skin_roi(image)

    return build_pixel_sequence(image)

In [6]:
def load_visual_dataset(image_dir, csv_path):
    df = pd.read_csv(csv_path)

    sequences = []
    labels = []

    for _, row in df.iterrows():
        img_path = os.path.join(image_dir, row['image_name'])
        seq = preprocess_image(img_path)

        sequences.append(seq)
        labels.append(row['TSB'])

    return np.array(sequences), np.array(labels, dtype=np.float32)

In [7]:
def load_visual_dataset(image_dir, csv_path):
    df = pd.read_csv(csv_path)

    sequences = []
    labels = []

    for _, row in df.iterrows():
        img_path = os.path.join(image_dir, row['image_idx'])  # FIXED COLUMN NAME
        seq = preprocess_image(img_path)

        sequences.append(seq)
        labels.append(row['blood(mg/dL)'])  # Target column

    return np.array(sequences), np.array(labels, dtype=np.float32)

In [8]:
# Check CSV file structure first
import pandas as pd

csv_path = "/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv"
df = pd.read_csv(csv_path)

print("CSV columns:")
print(df.columns.tolist())
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

CSV columns:
['patient_id', 'image_idx', 'gender', 'gestational_age', 'age(day)', 'weight', 'blood(mg/dL)', 'Treatment']

Dataset shape: (2235, 8)

First few rows:
   patient_id   image_idx gender  gestational_age  age(day)  weight  \
0           3  0003-1.jpg      F               40       5.2    3280   
1           3  0003-2.jpg      F               40       5.2    3280   
2           3  0003-3.jpg      F               40       5.2    3280   
3          35  0035-1.jpg      M               39       8.7    3760   
4          35  0035-2.jpg      M               39       8.7    3760   

   blood(mg/dL)  Treatment  
0           3.9          0  
1           3.9          0  
2           3.9          0  
3          12.2          0  
4          12.2          0  


In [9]:
image_dir = "/Users/shubhbhateja/College/minor_project/NeoJaundice/images"
csv_path = "/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv"

X, y = load_visual_dataset(image_dir, csv_path)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Total samples loaded:", X.shape[0])
print("Label count:", y.shape[0])

X shape: (2235, 4096, 6)
y shape: (2235,)
Total samples loaded: 2235
Label count: 2235


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
def build_visual_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv1D(32, 3, activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling1D(2)(x)

    x = tf.keras.layers.Conv1D(64, 3, activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)

    x = tf.keras.layers.Conv1D(128, 3, activation='relu')(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    output = tf.keras.layers.Dense(1)(x)

    model = tf.keras.Model(inputs, output)

    return model

In [12]:
model = build_visual_model(X_train.shape[1:])

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=1e-4,
    weight_decay=1e-5
)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4096, 6)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 4094, 32)       │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 2047, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 2045, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1022, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 1020, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,841 (155.63 KB)

 Trainable params: 39,841 (155.63 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early_stop = tf.keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=25
    ,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 53.1786 - mae: 5.7576 - val_loss: 24.7188 - val_mae: 4.0236
Epoch 2/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 26.8437 - mae: 4.2506 - val_loss: 21.6711 - val_mae: 3.7732
Epoch 3/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 25.4428 - mae: 4.1082 - val_loss: 19.8413 - val_mae: 3.5576
Epoch 4/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 23.5685 - mae: 3.9440 - val_loss: 19.4279 - val_mae: 3.5465
Epoch 5/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 22.8513 - mae: 3.8837 - val_loss: 18.5103 - val_mae: 3.4363
Epoch 6/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 22.2565 - mae: 3.8409 - val_loss: 17.1590 - val_mae: 3.2889
Epoch 7/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 20.2741 - mae: 3.6817 - val_loss: 18.6381 - val_mae: 3.3833
Epoch 8/25
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 22.6234 - mae: 3.8196 - val_loss: 15.9197 - val_mae: 3.1582
Epoch 9/25
101/101 ━━━━━━━━━━━━━

In [14]:
test_loss, test_mae = model.evaluate(X_test, y_test)

print("Test MAE:", test_mae)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 12.3580 - mae: 2.8893
Test MAE: 2.909555435180664


**Verification**

In [15]:
print("Total rows in CSV:", len(df))

Total rows in CSV: 2235


In [16]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2235, 4096, 6)
y shape: (2235,)


In [17]:
print("Unique patients:", df['patient_id'].nunique())

Unique patients: 745
